<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/workflow/physics_bounds_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deterministic physics and bounds checks between workflow steps

Tool calls and agent steps sometimes return numbers that cannot be physical: a probability of `1.8`, a negative mass, a speed faster than light, or a `NaN`. The next step will treat that payload as a fact unless something between the two steps rejects it.

The checker is a pure function: a payload goes in, a verdict comes out. It does not call a model. The workflow step that receives the tool event stays thin. It calls the checker and either stops with that verdict or emits a retry event. Scoring lives in the function, not in the workflow.

Every verdict has this shape:

```json
{"rule_table_version": "1.0.0", "verdict": "PASS", "violations": []}
```

`verdict` is `PASS`, `WARN`, or `FAIL`. This table emits only `PASS` and `FAIL`. `WARN` is reserved for a later soft rule. No violations map to `PASS`. A hard physics violation maps to `FAIL`. The same payload and the same table always recompute the same verdict.


## Install

The example uses `llama-index-core` only. The workflow runtime it re-exports supplies `@step`, events, retry policies, and `@catch_error`.


In [ ]:
%pip install llama-index-core -q

Workflow steps are async. Notebooks already have an event loop, so the cells below use `await` directly. In a script, wrap the same code in `asyncio.run()`.


In [5]:
import json
import math
from typing import Any

from llama_index.core.workflow import (
    Context,
    Event,
    StartEvent,
    StopEvent,
    Workflow,
    step,
)
from workflows import catch_error
from workflows.events import StepFailedEvent
from workflows.retry_policy import (
    retry_if_exception_type,
    retry_policy,
    stop_after_attempt,
    wait_none,
)


## Spec reference

The offline subset of [phm-lite-check](https://github.com/PHM-Audit/phm-lite-check) is the spec this example follows: `physics_text_check`, `physics_quantity_check`, and `robotics_data_check`. Those functions are a public reference for deterministic physics and quantity checks.

This notebook does not import that package. The rule table in the next cell is the whole policy, so a verdict can be recomputed from the payload and this page alone.


## A versioned rule table

`check_bounds` applies a bound only when the leaf key is listed. Unlisted quantities are left alone, except that every number must be finite.

* `probability` is a unit interval, `[0, 1]`. That bound is not applied to any other field.
* `mass_kg` is non-negative. A negative temperature delta is not a mass, so it is not rejected for its sign.
* `speed_m_s` is a magnitude limit. The checker compares `abs(value)` with `c`, which is the same as requiring the value to sit in `[-c, c]`. A large negative speed fails.
* `NaN` and both infinities fail closed, under rule id `finite`. A ruled field whose value is not a finite number fails closed under `malformed`. Neither case raises.

`version` is part of every verdict. Bump it when a bound changes. Keys are walked in sorted order, so two payloads with the same pairs produce the same violation list.


In [8]:
SPEED_OF_LIGHT_M_S = 299_792_458.0

# kind is there for readers. The checker itself uses min, max, and compare.
PHYSICS_BOUNDS_V1 = {
    "version": "1.0.0",
    "quantities": {
        "probability": {
            "kind": "unit_interval",
            "min": 0.0,
            "max": 1.0,
        },
        "mass_kg": {
            "kind": "non_negative",
            "min": 0.0,
            "max": None,
        },
        "speed_m_s": {
            "kind": "magnitude_max",
            "compare": "abs",
            "min": None,
            "max": SPEED_OF_LIGHT_M_S,
        },
    },
}

_VERDICT_KEYS = ("rule_table_version", "verdict", "violations")


def _is_number(value: Any) -> bool:
    # bool is a subclass of int; a flag is not a measurement.
    return isinstance(value, (int, float)) and not isinstance(value, bool)


def _render_number(value: int | float) -> int | float | str:
    if isinstance(value, float) and math.isnan(value):
        return "NaN"
    if isinstance(value, float) and math.isinf(value):
        return "Infinity" if value > 0 else "-Infinity"
    return value


def _render_value(value: Any) -> Any:
    if _is_number(value):
        return _render_number(value)
    try:
        rendered = repr(value)
    except Exception:
        rendered = type(value).__name__
    if len(rendered) > 80:
        return rendered[:77] + "..."
    return rendered


def _path_str(path: tuple[Any, ...]) -> str:
    text = ""
    for part in path:
        if isinstance(part, int):
            text += f"[{part}]"
        elif not text:
            text = str(part)
        else:
            text += f".{part}"
    return text


def _table_version(table: Any) -> str:
    if isinstance(table, dict) and isinstance(table.get("version"), str):
        return table["version"]
    return "unknown"


def _fail_closed(
    table: Any,
    reason: str,
    *,
    path: str = "",
    value: Any = None,
    rule_id: str = "malformed",
) -> dict[str, Any]:
    return {
        "rule_table_version": _table_version(table),
        "verdict": "FAIL",
        "violations": [
            {
                "rule_id": rule_id,
                "path": path,
                "value": value,
                "reason": reason,
            }
        ],
    }


def _bounds_reason(value: int | float, spec: dict[str, Any]) -> str | None:
    """Return a violation reason, or None when the value sits inside the rule."""
    high = spec.get("max")
    low = spec.get("min")
    if spec.get("compare") == "abs":
        magnitude = abs(float(value))
        if high is not None and magnitude > high:
            return f"abs(value) {magnitude} is above the maximum {high}"
        return None
    if low is not None and value < low:
        return f"value {value} is below the minimum {low}"
    if high is not None and value > high:
        return f"value {value} is above the maximum {high}"
    return None


def _walk(
    node: Any,
    path: tuple[Any, ...],
    quantities: dict[str, Any],
    violations: list[dict[str, Any]],
) -> None:
    if isinstance(node, dict):
        for key in sorted(node, key=lambda item: str(item)):
            _walk(node[key], path + (key,), quantities, violations)
        return
    if isinstance(node, list):
        for index, item in enumerate(node):
            _walk(item, path + (index,), quantities, violations)
        return

    location = _path_str(path)
    leaf = path[-1] if path else ""
    spec = quantities.get(leaf) if isinstance(leaf, str) else None

    if _is_number(node):
        if not math.isfinite(float(node)):
            violations.append(
                {
                    "rule_id": "finite",
                    "path": location,
                    "value": _render_number(node),
                    "reason": "value must be a finite number",
                }
            )
            return
        if not isinstance(spec, dict):
            return
        reason = _bounds_reason(node, spec)
        if reason is None:
            return
        violations.append(
            {
                "rule_id": leaf,
                "path": location,
                "value": _render_number(node),
                "reason": reason,
            }
        )
        return

    # A key the table names must be a finite number. Other strings and
    # labels are not measurements, so they are ignored.
    if isinstance(spec, dict):
        violations.append(
            {
                "rule_id": "malformed",
                "path": location,
                "value": _render_value(node),
                "reason": "ruled quantity must be a finite number",
            }
        )


def check_bounds(payload: Any, rules: dict[str, Any] | None = None) -> dict[str, Any]:
    """Return a verdict for ``payload``.

    Pure function of ``payload`` and ``rules``. Nested dicts and lists are
    walked. The function does not raise: a malformed payload, a non-finite
    number, or an unexpected error becomes ``verdict: FAIL``.
    """
    table: Any = PHYSICS_BOUNDS_V1 if rules is None else rules
    try:
        if not isinstance(table, dict) or not isinstance(table.get("quantities"), dict):
            return _fail_closed(table, "rule table is malformed")
        if not isinstance(payload, dict):
            return _fail_closed(
                table,
                "payload must be an object",
                value=_render_value(payload),
            )
        violations: list[dict[str, Any]] = []
        _walk(payload, (), table["quantities"], violations)
        return {
            "rule_table_version": table["version"],
            "verdict": "FAIL" if violations else "PASS",
            "violations": violations,
        }
    except Exception as exc:
        return _fail_closed(
            table,
            f"checker failed closed: {type(exc).__name__}",
        )


def show(verdict: dict[str, Any]) -> None:
    print(json.dumps(verdict, indent=2))


## Four false-positive traps

These calls do not start a workflow. Each one is a check the rule table has to get right. A tool-shaped payload uses the same keys a `ToolCall` (`tool_kwargs`) and a `ToolCallResult` (`tool_output`) would contribute. The checker only sees the dict.

1. A legitimate negative quantity passes. `temperature_delta_c = -3` is physically fine. Only keys the table marks non-negative, such as `mass_kg`, fail for being negative.
2. The unit interval applies only to `probability`. A count of `50` or a temperature delta of `1.8` is not a percentage.
3. Speed uses `abs(value)`. `-2c` fails, and so does `+2c`. A negative but sub-luminal speed passes.
4. `NaN`, infinities, and malformed values fail closed. The call returns a `FAIL` verdict. It does not raise.


In [10]:
def expect(name: str, payload: Any, verdict_name: str) -> dict[str, Any]:
    """Print a verdict and raise if it is not the expected PASS or FAIL."""
    got = check_bounds(payload)
    print(name)
    show(got)
    if set(got) != set(_VERDICT_KEYS):
        raise AssertionError(f"{name}: unexpected keys {sorted(got)}")
    if got["verdict"] not in {"PASS", "WARN", "FAIL"}:
        raise AssertionError(f"{name}: bad verdict {got['verdict']!r}")
    if got["verdict"] != verdict_name:
        raise AssertionError(f"{name}: expected {verdict_name}, got {got['verdict']}")
    return got


legal = {"probability": 0.25, "mass_kg": 1.5, "speed_m_s": 10.0}

expect(
    "negative temperature delta passes",
    {"tool_output": {**legal, "temperature_delta_c": -3}},
    "PASS",
)

mass_and_temperature = expect(
    "negative mass fails, negative temperature does not",
    {"tool_output": {**legal, "temperature_delta_c": -3, "mass_kg": -2.0}},
    "FAIL",
)
mass_paths = [item["path"] for item in mass_and_temperature["violations"]]
if mass_paths != ["tool_output.mass_kg"]:
    raise AssertionError(f"mass case flagged extra paths: {mass_paths}")

expect(
    "unit interval is not applied to other fields",
    {
        "tool_output": {
            **legal,
            "count": 50,
            "temperature_delta_c": 1.8,
        }
    },
    "PASS",
)

probability = expect(
    "probability above 1 fails",
    {
        "tool_output": {
            **legal,
            "count": 50,
            "temperature_delta_c": 1.8,
            "probability": 1.8,
        }
    },
    "FAIL",
)
probability_rules = [item["rule_id"] for item in probability["violations"]]
if probability_rules != ["probability"]:
    raise AssertionError(f"probability case flagged {probability_rules}")

expect(
    "sub-luminal negative speed passes",
    {"tool_output": {**legal, "speed_m_s": -10.0}},
    "PASS",
)

negative_fast = expect(
    "negative superluminal speed fails",
    {"tool_output": {**legal, "speed_m_s": -2 * SPEED_OF_LIGHT_M_S}},
    "FAIL",
)
if negative_fast["violations"][0]["rule_id"] != "speed_m_s":
    raise AssertionError(negative_fast)
if "abs(value)" not in negative_fast["violations"][0]["reason"]:
    raise AssertionError(negative_fast["violations"][0]["reason"])

expect(
    "positive superluminal speed fails",
    {"tool_output": {**legal, "speed_m_s": 2 * SPEED_OF_LIGHT_M_S}},
    "FAIL",
)


def fails_closed(name: str, payload: Any) -> dict[str, Any]:
    try:
        got = check_bounds(payload)
    except Exception as exc:
        raise AssertionError(f"{name} crashed the checker: {exc}") from exc
    print(name)
    show(got)
    if got["verdict"] != "FAIL":
        raise AssertionError(f"{name}: expected FAIL, got {got['verdict']}")
    return got


fails_closed("NaN mass fails closed", {"tool_output": {"mass_kg": float("nan")}})
fails_closed(
    "infinite speed fails closed",
    {"tool_output": {"speed_m_s": float("inf")}},
)
fails_closed(
    "negative infinity fails closed",
    {"tool_output": {"speed_m_s": float("-inf")}},
)
fails_closed(
    "NaN on an unlisted quantity fails closed",
    {"tool_output": {"temperature_delta_c": float("nan")}},
)
fails_closed(
    "non-numeric probability fails closed",
    {"tool_output": {**legal, "probability": "not-a-number"}},
)
fails_closed(
    "null mass fails closed",
    {"tool_output": {**legal, "mass_kg": None}},
)
fails_closed("non-object payload fails closed", None)
fails_closed("list payload fails closed", [legal])

print(
    "recomputable:",
    check_bounds({"tool_output": {**legal, "probability": 1.8}})
    == check_bounds({"tool_output": {**legal, "probability": 1.8}}),
)


negative temperature delta passes
{
  "rule_table_version": "1.0.0",
  "verdict": "PASS",
  "violations": []
}
negative mass fails, negative temperature does not
{
  "rule_table_version": "1.0.0",
  "verdict": "FAIL",
  "violations": [
    {
      "rule_id": "mass_kg",
      "path": "tool_output.mass_kg",
      "value": -2.0,
      "reason": "value -2.0 is below the minimum 0.0"
    }
  ]
}
unit interval is not applied to other fields
{
  "rule_table_version": "1.0.0",
  "verdict": "PASS",
  "violations": []
}
probability above 1 fails
{
  "rule_table_version": "1.0.0",
  "verdict": "FAIL",
  "violations": [
    {
      "rule_id": "probability",
      "path": "tool_output.probability",
      "value": 1.8,
      "reason": "value 1.8 is above the maximum 1.0"
    }
  ]
}
sub-luminal negative speed passes
{
  "rule_table_version": "1.0.0",
  "verdict": "PASS",
  "violations": []
}
negative superluminal speed fails
{
  "rule_table_version": "1.0.0",
  "verdict": "FAIL",
  "violations": [
 

## The workflow

Two steps, and no model. The handler does not score numbers itself.

1. `read_tool_output` stands in for a tool call. It emits the next scripted payload as a `ToolObservationEvent`. The list of payloads is fixed, so the run does not depend on a sampler.
2. `validate_bounds` calls `check_bounds` and writes a `PhysicsVerdictEvent` to the stream. `FAIL` stops the run, or emits `PhysicsRetryEvent` when `on_violation` is `"retry"` and attempts remain. `PASS` stops with that verdict. `WARN` would stop the same way as `PASS`. This table does not emit `WARN`.

If the checker ever raised, the step would still return a `FAIL` verdict instead of crashing the run.


In [12]:
class ToolObservationEvent(Event):
    """Numeric payload from a tool or agent step."""

    tool_name: str
    payload: dict
    attempt: int


class PhysicsRetryEvent(Event):
    """Ask the reader for another payload after a FAIL verdict."""

    tool_name: str
    attempt: int
    violations: list


class PhysicsVerdictEvent(Event):
    """Copy of the verdict, published on the handler stream."""

    rule_table_version: str
    verdict: str
    violations: list
    tool_name: str
    attempt: int


def _handler_fail_closed(exc: BaseException) -> dict[str, Any]:
    return {
        "rule_table_version": PHYSICS_BOUNDS_V1["version"],
        "verdict": "FAIL",
        "violations": [
            {
                "rule_id": "malformed",
                "path": "",
                "value": None,
                "reason": f"checker failed closed: {type(exc).__name__}",
            }
        ],
    }


class PhysicsBoundsWorkflow(Workflow):
    def __init__(
        self,
        *,
        readings: list[dict],
        on_violation: str = "stop",
        max_attempts: int = 3,
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        if on_violation not in {"stop", "retry"}:
            raise ValueError("on_violation must be 'stop' or 'retry'")
        if not readings:
            raise ValueError("readings must contain at least one payload")
        self._readings = readings
        self._on_violation = on_violation
        self._max_attempts = max_attempts

    @step
    async def read_tool_output(
        self, ctx: Context, ev: StartEvent | PhysicsRetryEvent
    ) -> ToolObservationEvent:
        attempt = 0 if isinstance(ev, StartEvent) else ev.attempt
        index = min(attempt, len(self._readings) - 1)
        observed = ToolObservationEvent(
            tool_name=ev.tool_name,
            payload=self._readings[index],
            attempt=attempt,
        )
        ctx.write_event_to_stream(observed)
        return observed

    @step
    async def validate_bounds(
        self, ctx: Context, ev: ToolObservationEvent
    ) -> PhysicsRetryEvent | StopEvent:
        try:
            verdict = check_bounds(ev.payload)
        except Exception as exc:
            verdict = _handler_fail_closed(exc)
        ctx.write_event_to_stream(
            PhysicsVerdictEvent(
                rule_table_version=verdict["rule_table_version"],
                verdict=verdict["verdict"],
                violations=verdict["violations"],
                tool_name=ev.tool_name,
                attempt=ev.attempt,
            )
        )
        failed = verdict["verdict"] == "FAIL"
        should_stop = (
            not failed
            or self._on_violation == "stop"
            or ev.attempt + 1 >= self._max_attempts
        )
        if should_stop:
            action = "stop" if failed else "accept"
            return StopEvent(
                result={
                    **verdict,
                    "tool_name": ev.tool_name,
                    "attempt": ev.attempt,
                    "action": action,
                }
            )
        return PhysicsRetryEvent(
            tool_name=ev.tool_name,
            attempt=ev.attempt + 1,
            violations=verdict["violations"],
        )


## Stop or accept at the step boundary

Each run uses a fresh workflow. The printed result is the checker verdict plus which tool produced it. `action` is `"accept"` for `PASS` and `"stop"` for `FAIL`.


In [14]:
async def run_bounds(
    readings: list[dict],
    *,
    tool_name: str,
    on_violation: str = "stop",
    max_attempts: int = 3,
) -> dict:
    workflow = PhysicsBoundsWorkflow(
        readings=readings,
        on_violation=on_violation,
        max_attempts=max_attempts,
        timeout=30,
    )
    handler = workflow.run(tool_name=tool_name)
    async for event in handler.stream_events():
        if isinstance(event, ToolObservationEvent):
            print(f"stream observe attempt={event.attempt} tool={event.tool_name}")
        elif isinstance(event, PhysicsVerdictEvent):
            rules = [item["rule_id"] for item in event.violations]
            print(
                f"stream verdict attempt={event.attempt} "
                f"verdict={event.verdict} rules={rules}"
            )
    return await handler


In [15]:
stop_cases = {
    "negative temperature": {"tool_output": {**legal, "temperature_delta_c": -3}},
    "negative mass": {"tool_output": {**legal, "mass_kg": -2.0}},
    "probability": {"tool_output": {**legal, "probability": 1.8, "count": 50}},
    "negative superluminal": {
        "tool_output": {**legal, "speed_m_s": -2 * SPEED_OF_LIGHT_M_S}
    },
    "non-finite": {"tool_output": {"mass_kg": float("nan"), "speed_m_s": float("inf")}},
    "malformed": {"tool_output": {**legal, "probability": "not-a-number"}},
}
for name, payload in stop_cases.items():
    print("==", name)
    result = await run_bounds([payload], tool_name=f"{name}_sensor")
    show(result)
    if result["verdict"] not in {"PASS", "WARN", "FAIL"}:
        raise AssertionError(result)


== negative temperature
stream observe attempt=0 tool=negative temperature_sensor
stream verdict attempt=0 verdict=PASS rules=[]
{
  "rule_table_version": "1.0.0",
  "verdict": "PASS",
  "violations": [],
  "tool_name": "negative temperature_sensor",
  "attempt": 0,
  "action": "accept"
}
== negative mass
stream observe attempt=0 tool=negative mass_sensor
stream verdict attempt=0 verdict=FAIL rules=['mass_kg']
{
  "rule_table_version": "1.0.0",
  "verdict": "FAIL",
  "violations": [
    {
      "rule_id": "mass_kg",
      "path": "tool_output.mass_kg",
      "value": -2.0,
      "reason": "value -2.0 is below the minimum 0.0"
    }
  ],
  "tool_name": "negative mass_sensor",
  "attempt": 0,
  "action": "stop"
}
== probability
stream observe attempt=0 tool=probability_sensor
stream verdict attempt=0 verdict=FAIL rules=['probability']
{
  "rule_table_version": "1.0.0",
  "verdict": "FAIL",
  "violations": [
    {
      "rule_id": "probability",
      "path": "tool_output.probability",
  

## Emit a retry event instead of stopping

The first reading is faster than light. The second is allowed. With `on_violation="retry"` the handler sends `PhysicsRetryEvent`, the reader emits the next scripted payload, and the run ends when the verdict is `PASS`.

A second run never gets a legal reading. `max_attempts=2` stops it with the last `FAIL` rather than looping.


In [17]:
corrected = {"tool_output": dict(legal)}
print("== retry then accept")
show(
    await run_bounds(
        [
            {"tool_output": {**legal, "speed_m_s": 2 * SPEED_OF_LIGHT_M_S}},
            corrected,
        ],
        tool_name="speed_sensor",
        on_violation="retry",
        max_attempts=3,
    )
)

print("== retry exhausted")
show(
    await run_bounds(
        [
            {"tool_output": {**legal, "probability": 1.8}},
            {"tool_output": {**legal, "probability": 1.8}},
        ],
        tool_name="probability_sensor",
        on_violation="retry",
        max_attempts=2,
    )
)


== retry then accept
stream observe attempt=0 tool=speed_sensor
stream verdict attempt=0 verdict=FAIL rules=['speed_m_s']
stream observe attempt=1 tool=speed_sensor
stream verdict attempt=1 verdict=PASS rules=[]
{
  "rule_table_version": "1.0.0",
  "verdict": "PASS",
  "violations": [],
  "tool_name": "speed_sensor",
  "attempt": 1,
  "action": "accept"
}
== retry exhausted
stream observe attempt=0 tool=probability_sensor
stream verdict attempt=0 verdict=FAIL rules=['probability']
stream observe attempt=1 tool=probability_sensor
stream verdict attempt=1 verdict=FAIL rules=['probability']
{
  "rule_table_version": "1.0.0",
  "verdict": "FAIL",
  "violations": [
    {
      "rule_id": "probability",
      "path": "tool_output.probability",
      "value": 1.8,
      "reason": "value 1.8 is above the maximum 1.0"
    }
  ],
  "tool_name": "probability_sensor",
  "attempt": 1,
  "action": "stop"
}


## Compose the same check with a retry policy

`retry_policy` retries a step when it raises. That is a different knob from the retry event above.

* `TransientSensorError` means the tool returned nothing. The policy retries it, with no delay, up to 3 attempts.
* `PhysicsViolationError` means `check_bounds` returned `FAIL`. The policy does not retry that exception. Retrying would call the same tool again on data that is already known to be impossible.

`@catch_error` turns the exception into a `StopEvent`. The result still uses `verdict` and `violations`. Build a new workflow instance per run: the script cursor lives on `self`.


In [19]:
class TransientSensorError(Exception):
    """The tool returned no payload. Calling it again may succeed."""


class PhysicsViolationError(Exception):
    """The payload is complete and the rule table returned FAIL."""

    def __init__(self, verdict: dict[str, Any]) -> None:
        self.verdict = verdict
        rule_ids = ", ".join(item["rule_id"] for item in verdict["violations"])
        super().__init__(
            f"physics violation ({verdict['rule_table_version']}): {rule_ids}"
        )


# Retry only empty reads. A physics FAIL falls through on the first attempt.
TRANSIENT_READ_POLICY = retry_policy(
    retry=retry_if_exception_type(TransientSensorError),
    wait=wait_none(),
    stop=stop_after_attempt(3),
)


class RetryPolicyPhysicsWorkflow(Workflow):
    def __init__(self, script: list[dict | None], **kwargs: Any) -> None:
        super().__init__(**kwargs)
        self._script = list(script)
        self._index = 0

    @step(retry_policy=TRANSIENT_READ_POLICY)
    async def read_and_check(self, ev: StartEvent) -> StopEvent:
        item = self._script[self._index]
        self._index += 1
        if item is None:
            raise TransientSensorError("tool returned no payload")
        verdict = check_bounds(item)
        if verdict["verdict"] == "FAIL":
            raise PhysicsViolationError(verdict)
        return StopEvent(
            result={**verdict, "action": "accept", "attempt": self._index - 1}
        )

    @catch_error(for_steps=["read_and_check"])
    async def on_check_failed(self, ev: StepFailedEvent) -> StopEvent:
        error = ev.exception
        carried = getattr(error, "verdict", None)
        if isinstance(carried, dict) and "verdict" in carried:
            result = {
                **carried,
                "action": "stop",
                "attempts": ev.attempts,
                "error_type": type(error).__name__,
            }
        else:
            result = {
                "rule_table_version": None,
                "verdict": "FAIL",
                "violations": [
                    {
                        "rule_id": "transient_read",
                        "path": "",
                        "value": None,
                        "reason": "tool returned no payload",
                    }
                ],
                "action": "stop",
                "attempts": ev.attempts,
                "error_type": type(error).__name__,
            }
        return StopEvent(result=result)


async def run_retry_policy(script: list[dict | None]) -> dict:
    workflow = RetryPolicyPhysicsWorkflow(script, timeout=30)
    return await workflow.run(tool_name="sensor")


print("== transient reads, then a legal payload")
show(await run_retry_policy([None, None, {"tool_output": dict(legal)}]))

print("== probability 1.8 is not retried")
show(
    await run_retry_policy(
        [{"tool_output": {**legal, "probability": 1.8}}],
    )
)

print("== empty reads exhausted")
show(await run_retry_policy([None, None, None]))


== transient reads, then a legal payload
{
  "rule_table_version": "1.0.0",
  "verdict": "PASS",
  "violations": [],
  "action": "accept",
  "attempt": 2
}
== probability 1.8 is not retried
{
  "rule_table_version": "1.0.0",
  "verdict": "FAIL",
  "violations": [
    {
      "rule_id": "probability",
      "path": "tool_output.probability",
      "value": 1.8,
      "reason": "value 1.8 is above the maximum 1.0"
    }
  ],
  "action": "stop",
  "attempts": 1,
  "error_type": "PhysicsViolationError"
}
== empty reads exhausted
{
  "rule_table_version": null,
  "verdict": "FAIL",
  "violations": [
    {
      "rule_id": "transient_read",
      "path": "",
      "value": null,
      "reason": "tool returned no payload"
    }
  ],
  "action": "stop",
  "attempts": 3,
  "error_type": "TransientSensorError"
}


## Bump the table version when a bound changes

A probability of `0.7` is `PASS` on version `1.0.0`. A copied table that caps probability at `0.5` returns `FAIL`, and the verdict names `1.1.0`. Recomputing with the original table is still `PASS`. The handler code does not change.


In [21]:
reading = {"probability": 0.7, "mass_kg": 1.0, "speed_m_s": 10.0}
stricter = {
    "version": "1.1.0",
    "quantities": {
        **PHYSICS_BOUNDS_V1["quantities"],
        "probability": {"kind": "unit_interval", "min": 0.0, "max": 0.5},
    },
}
print("table 1.0.0")
show(check_bounds(reading))
print("table 1.1.0")
show(check_bounds(reading, stricter))


table 1.0.0
{
  "rule_table_version": "1.0.0",
  "verdict": "PASS",
  "violations": []
}
table 1.1.0
{
  "rule_table_version": "1.1.0",
  "verdict": "FAIL",
  "violations": [
    {
      "rule_id": "probability",
      "path": "probability",
      "value": 0.7,
      "reason": "value 0.7 is above the maximum 0.5"
    }
  ]
}


## What to keep

Call `check_bounds` at the step that receives the tool or agent event. Return its verdict from `StopEvent` when the run must end. Emit a separate event when another step is allowed to try again. Use a retry policy only for failures that might succeed on another call, and keep a `FAIL` from the rule table out of that policy.

Guard the four traps in the table itself: do not ban every negative, do not put a percentage bound on an unrelated field, compare speed with `abs(value)`, and fail closed on `NaN`, infinity, and malformed values.

The rule table in this notebook is the whole policy. [phm-lite-check](https://github.com/PHM-Audit/phm-lite-check) is the offline spec those rules follow. Change a number here, bump `version`, and `check_bounds` will reproduce every verdict from the payload plus that table.
